# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ravikiranbathe/flyrank-ai/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
!git clone https://github.com/Ravikiranbathe/flyrank-ai.git

%cd flyrank-ai

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)

Cloning into 'flyrank-ai'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 149 (delta 57), reused 101 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (149/149), 1.86 MiB | 5.90 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/flyrank-ai/flyrank-ai
(30000, 44)


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method Choice

For this project, I chose the **Random Forest Classifier** because it works well with multiple search performance signals and can capture complex patterns in the data. It also provides feature importance, making it easier to understand which features influence the predictions.

I selected this method to compare it fairly with my Week 4 baseline using the same dataset, split, and evaluation metrics to see whether it improves the results.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split Design

I used the same dataset as my Week 4 baseline so the comparison is fair. The data is divided into training and testing sets, with the model trained on the training data and evaluated on the test data. Using the same split and evaluation process ensures that any improvement comes from the model rather than differences in the data.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Train and Compare

The Random Forest model is trained using the same dataset as the Week 4 baseline. The baseline is a rule-based scoring system, while the Random Forest learns patterns from the data. Both approaches are evaluated on the same data so their performance can be compared fairly.


In [10]:
import numpy as np

# Recreate Week 4 baseline
df["baseline_score"] = 0

df.loc[df["content_age_days"] >= 365, "baseline_score"] += 40
df.loc[df["trend_pct"] < -10, "baseline_score"] += 25
df.loc[df["ctr"] < 2, "baseline_score"] += 20
df.loc[df["impressions_90d"] >= 1000, "baseline_score"] += 15

# Create target
df["target"] = (df["baseline_score"] >= 60).astype(int)

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Create target from the baseline rule
df["target"] = (df["baseline_score"] >= 60).astype(int)

# Select features
features = [
    "content_age_days",
    "trend_pct",
    "ctr",
    "impressions_90d"
]

X = df[features]
y = df["target"]

# Handle missing values
X = X.fillna(0)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Train Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

# Predictions
y_pred = rf.predict(X_test)

# Metrics
results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Random Forest": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred)
    ]
})

comparison = pd.DataFrame({
    "Model": ["Week 4 Baseline", "Random Forest"],
    "Accuracy": ["Rule-based", round(accuracy_score(y_test, y_pred), 4)],
    "Precision": ["Rule-based", round(precision_score(y_test, y_pred), 4)],
    "Recall": ["Rule-based", round(recall_score(y_test, y_pred), 4)],
    "F1 Score": ["Rule-based", round(f1_score(y_test, y_pred), 4)]
})

comparison

,Model,Accuracy,Precision,Recall,F1 Score
0,Week 4 Baseline,Rule-based,Rule-based,Rule-based,Rule-based
1,Random Forest,0.9998,0.9996,1.0,0.9998


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Errors and Interpretation

The Random Forest model performed well because it learned the same patterns used in the Week 4 baseline.

The most important features were **content_age_days**, **impressions_90d**, and **trend_pct**, while **CTR** had the lowest importance.

The model may still make incorrect predictions for pages affected by factors that are not included in the selected features.

In [12]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("Feature Importance")
display(importance)

Feature Importance


,Feature,Importance
0,content_age_days,0.368995
3,impressions_90d,0.332206
1,trend_pct,0.238058
2,ctr,0.060741


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.